# 01. Profiling de Archivos de Obras Públicas - INFOBRAS (CGR)

Este notebook realiza el **profiling estructural y de calidad de datos** de un archivo descargado de la Contraloría General de la República (INFOBRAS), admitiendo formatos **`.xlsx`** y **`.csv`**.

### Funcionalidades principales:
1. **Parámetro de entrada**: Ruta relativa del archivo respecto a la raíz del proyecto (`ARCHIVO_RELATIVO`) y número de fila donde inicia la cabecera (`FILA_CABECERA = 4`, saltando las primeras 3 filas de metadatos del reporte).
2. **Lectura adaptable (`.xlsx` / `.csv`)**: Soporta archivos con distinta estructura de columnas, extrae los metadatos superiores (filas 1 a 3) y detecta automáticamente el separador/codificación en caso de archivos `.csv`.
3. **Profiling integral**:
   - Cantidad total de registros (filas) y columnas.
   - Distribución de tipos de datos (`dtype` de pandas y tipo inferido real sobre valores no nulos).
   - Listado completo de todas las columnas con su posición, nombre original, nombre normalizado (`snake_case`), tipo de dato, conteo y porcentaje de nulos/vacíos, cardinalidad (valores únicos) y ejemplos de valores.

## 1. Configuración e Importación de Librerías

In [1]:
import csv
import re
import unicodedata
from pathlib import Path
import pandas as pd
import numpy as np
import openpyxl

# Configuración de visualización en Pandas para mostrar todas las columnas en el listado
pd.set_option('display.max_rows', 200)
pd.set_option('display.max_columns', 50)
pd.set_option('display.max_colwidth', 80)
pd.set_option('display.width', 1200)

def obtener_raiz_proyecto() -> Path:
    """Detecta la raíz del proyecto buscando carpetas clave (etl, data, configs)."""
    actual = Path.cwd().resolve()
    for carpeta in [actual, *actual.parents]:
        if (carpeta / 'data').exists() or (carpeta / 'etl').exists():
            return carpeta
    return actual

PROJECT_ROOT = obtener_raiz_proyecto()
print(f'Raíz del proyecto detectada: {PROJECT_ROOT}')

Raíz del proyecto detectada: C:\ws\formacion\uni-mia\mia-tesis-opa


## 2. Parámetros del Archivo a Evaluar

Defina en `ARCHIVO_RELATIVO` la ruta relativa desde la raíz del proyecto hacia el archivo `.xlsx` o `.csv` que desea perfilar, y en `FILA_CABECERA` la fila donde se encuentran los nombres de las columnas (por defecto `4` para los reportes de INFOBRAS).

In [19]:
# ==============================================================================
# PARÁMETROS DE ENTRADA
# ==============================================================================
ARCHIVO_RELATIVO = "data/raw/cgr/2025/DataSet-Obras-Publicas 01-10-2025.xlsx"  # Puede ser .xlsx o .csv
FILA_CABECERA = 4  # La cabecera inicia en la fila 4 (se saltan las primeras 3 filas)
# ==============================================================================

ruta_archivo = (PROJECT_ROOT / ARCHIVO_RELATIVO).resolve()
if not ruta_archivo.exists():
    raise FileNotFoundError(f'No se encontró el archivo en la ruta: {ruta_archivo}')

print(f'Archivo seleccionado : {ARCHIVO_RELATIVO}')
print(f'Ruta absoluta        : {ruta_archivo}')
print(f'Extensión            : {ruta_archivo.suffix.lower()}')
print(f'Tamaño del archivo   : {ruta_archivo.stat().st_size / (1024 * 1024):.2f} MB ({ruta_archivo.stat().st_size:,} bytes)')

Archivo seleccionado : data/raw/cgr/2025/DataSet-Obras-Publicas 01-10-2025.xlsx
Ruta absoluta        : C:\ws\formacion\uni-mia\mia-tesis-opa\data\raw\cgr\2025\DataSet-Obras-Publicas 01-10-2025.xlsx
Extensión            : .xlsx
Tamaño del archivo   : 58.12 MB (60,945,412 bytes)


## 3. Funciones de Carga Flexible (`.xlsx` / `.csv`) y Profiling

- Extrae las filas previas a la cabecera (`filas 1..3`) para conservar la trazabilidad de la fecha de consulta del reporte.
- Salta automáticamente las filas previas (`skiprows = fila_cabecera - 1`) para ubicar la cabecera en la fila 4.
- Soporta cualquier variación en la cantidad o nombres de columnas entre distintos cortes mensuales.

In [20]:
def normalizar_nombre_columna(nombre: str) -> str:
    """Convierte un nombre de columna a snake_case sin tildes ni caracteres especiales."""
    texto = str(nombre).strip()
    texto = unicodedata.normalize('NFKD', texto).encode('ascii', 'ignore').decode('ascii')
    texto = re.sub(r'[^a-zA-Z0-9]+', '_', texto).strip('_').lower()
    return texto or 'columna_sin_nombre'


def extraer_metadatos_cabecera(ruta: Path, fila_cabecera: int = 4) -> list[str]:
    """Lee las filas previas a la cabecera (ej. filas 1 a 3) para extraer metadatos del archivo."""
    filas_previas = max(0, fila_cabecera - 1)
    if filas_previas == 0:
        return []

    ext = ruta.suffix.lower()
    metadatos = []
    try:
        if ext in ('.xlsx', '.xls'):
            wb = openpyxl.load_workbook(ruta, read_only=True, data_only=True)
            ws = wb.active
            for i, row in enumerate(ws.iter_rows(max_row=filas_previas, values_only=True), start=1):
                valores = [str(c).strip() for c in row if c is not None and str(c).strip() != '']
                metadatos.append(f"Fila {i}: {' | '.join(valores[:5]) if valores else '(vacía)'}")
            wb.close()
        elif ext == '.csv':
            for enc in ('utf-8-sig', 'utf-8', 'latin-1', 'cp1252'):
                try:
                    with open(ruta, 'r', encoding=enc, errors='strict') as f:
                        for i in range(1, filas_previas + 1):
                            linea = f.readline().strip()
                            metadatos.append(f'Fila {i}: {linea[:120]}')
                    break
                except UnicodeDecodeError:
                    continue
    except Exception as e:
        metadatos.append(f'No se pudieron extraer metadatos previos: {e}')
    return metadatos


def cargar_archivo_cgr(ruta: Path, fila_cabecera: int = 4) -> pd.DataFrame:
    """
    Carga un archivo .xlsx o .csv de la Contraloría saltando las filas previas a la cabecera.
    Por defecto fila_cabecera=4 (skiprows=3).
    """
    skiprows = max(0, fila_cabecera - 1)
    ext = ruta.suffix.lower()

    if ext in ('.xlsx', '.xls'):
        df = pd.read_excel(ruta, skiprows=skiprows)
    elif ext == '.csv':
        df = None
        ultimo_error = None
        for enc in ('utf-8-sig', 'utf-8', 'latin-1', 'cp1252'):
            try:
                with open(ruta, 'r', encoding=enc) as f:
                     Lineas = [f.readline() for _ in range(fila_cabecera + 5)]
                muestra = ''.join(Lineas[skiprows:])
                try:
                    dialect = csv.Sniffer().sniff(muestra, delimiters=',;|\t')
                    sep = dialect.delimiter
                except Exception:
                    sep = ','

                df_temp = pd.read_csv(ruta, skiprows=skiprows, encoding=enc, sep=sep, low_memory=False)
                # Si el CSV ya fue exportado sin las 3 filas superiores y skiprows saltó datos por error
                if df_temp.shape[1] == 1 and skiprows > 0:
                    df_temp = pd.read_csv(ruta, skiprows=0, encoding=enc, sep=sep, low_memory=False)
                df = df_temp
                break
            except Exception as e:
                ultimo_error = e
        if df is None:
            raise RuntimeError(f'Error al leer el archivo CSV {ruta}: {ultimo_error}')
    else:
        raise ValueError(f'Extensión no soportada: {ext}. Use .xlsx o .csv')

    # Limpiar espacios en blanco en los nombres de columnas
    df.columns = [str(col).strip() for col in df.columns]
    return df


def inferir_tipo_contenido(serie: pd.Series) -> str:
    """Infiere el tipo de dato semántico de una serie evaluando sus valores no nulos."""
    s_limpia = serie.dropna()
    if s_limpia.empty:
        return 'vacío (100% nulo)'

    # Si es object/string, descartar cadenas vacías
    if pd.api.types.is_object_dtype(s_limpia) or pd.api.types.is_string_dtype(s_limpia):
        s_str = s_limpia.astype(str).str.strip()
        s_str = s_str[s_str != '']
        if s_str.empty:
            return 'vacío (cadenas vacías)'

        # Verificar si es booleano Sí/No
        valores_upper = set(s_str.str.upper().unique())
        if valores_upper.issubset({'SI', 'SÍ', 'NO', 'S', 'N'}):
            return 'booleano (SI/NO)'

        # Verificar si es numérico almacenado como texto
        muestra = s_str.head(500)
        num_conv = pd.to_numeric(muestra.str.replace(',', '', regex=False), errors='coerce')
        if num_conv.notna().mean() >= 0.95:
            # Distinguir códigos con ceros a la izquierda (ej. '0608')
            if muestra.str.match(r'^0\d+$').any():
                return 'código numérico (texto con ceros izq.)'
            if (num_conv % 1 == 0).all():
                return 'entero (almacenado como texto)'
            return 'decimal (almacenado como texto)'

        # Verificar si es fecha en formato DD/MM/YYYY o YYYY-MM-DD
        if muestra.str.match(r'^(\d{2}/\d{2}/\d{4}|\d{4}-\d{2}-\d{2})').mean() >= 0.85:
            return 'fecha (almacenada como texto)'

        return 'texto / categórico'

    if pd.api.types.is_bool_dtype(serie):
        return 'booleano'
    if pd.api.types.is_integer_dtype(serie):
        return 'entero (int)'
    if pd.api.types.is_float_dtype(serie):
        if (s_limpia % 1 == 0).all():
            return 'entero (float por nulos)'
        return 'decimal (float)'
    if pd.api.types.is_datetime64_any_dtype(serie):
        return 'fecha (datetime)'

    return str(serie.dtype)


def generar_profiling_columnas(df: pd.DataFrame) -> pd.DataFrame:
    """Genera una tabla detallada con el profiling de cada columna del DataFrame."""
    total_filas = len(df)
    registros_perfil = []

    for idx, col in enumerate(df.columns, start=1):
        serie = df[col]
        # Tratar cadenas vacías como nulos para el conteo real de vacíos
        if pd.api.types.is_object_dtype(serie) or pd.api.types.is_string_dtype(serie):
            mask_vacio = serie.isna() | (serie.astype(str).str.strip() == '')
            serie_no_vacia = serie[~mask_vacio]
        else:
            mask_vacio = serie.isna()
            serie_no_vacia = serie.dropna()

        nulos = int(mask_vacio.sum())
        no_nulos = total_filas - nulos
        pct_nulos = round((nulos / total_filas) * 100, 2) if total_filas > 0 else 0.0
        unicos = int(serie_no_vacia.nunique(dropna=True))
        pct_unicos = round((unicos / no_nulos) * 100, 2) if no_nulos > 0 else 0.0

        ejemplos = serie_no_vacia.drop_duplicates().head(3).tolist()
        muestra_str = ' | '.join(str(x)[:45] for x in ejemplos) if ejemplos else '-'

        registros_perfil.append({
            'nro_col': idx,
            'nombre_columna': col,
            'columna_normalizada': normalizar_nombre_columna(col),
            'dtype_pandas': str(serie.dtype),
            'tipo_inferido': inferir_tipo_contenido(serie),
            'total_registros': total_filas,
            'no_nulos': no_nulos,
            'nulos_o_vacios': nulos,
            'pct_nulos (%)': pct_nulos,
            'valores_unicos': unicos,
            'pct_cardinalidad (%)': pct_unicos,
            'muestra_valores': muestra_str,
        })

    return pd.DataFrame(registros_perfil)

## 4. Carga del Archivo y Resumen General (Cantidad de Registros y Estructura)

In [21]:
# 1. Extraer metadatos del encabezado (Filas 1 a 3)
metadatos_cabecera = extraer_metadatos_cabecera(ruta_archivo, fila_cabecera=FILA_CABECERA)
print('=== METADATOS DEL REPORTE (FILAS PREVIAS A LA CABECERA) ===')
for linea in metadatos_cabecera:
    print(' ', linea)

# 2. Cargar el dataset saltando hasta la fila 4
print(f'\nCargando datos desde {ARCHIVO_RELATIVO} (Cabecera en fila {FILA_CABECERA})...')
df = cargar_archivo_cgr(ruta_archivo, fila_cabecera=FILA_CABECERA)

# 3. Resumen ejecutivo del dataset
total_registros, total_columnas = df.shape
memoria_mb = df.memory_usage(deep=True).sum() / (1024 * 1024)
duplicados_exactos = int(df.duplicated().sum())

resumen_general = pd.DataFrame([
    {'Métrica': 'Archivo (Ruta Relativa)', 'Valor': ARCHIVO_RELATIVO},
    {'Métrica': 'Formato / Extensión', 'Valor': ruta_archivo.suffix.lower()},
    {'Métrica': 'Fila de Cabecera Utilizada', 'Valor': f'Fila {FILA_CABECERA} (skiprows={FILA_CABECERA - 1})'},
    {'Métrica': 'Cantidad Total de Registros (Filas)', 'Valor': f'{total_registros:,}'},
    {'Métrica': 'Cantidad Total de Columnas', 'Valor': f'{total_columnas:,}'},
    {'Métrica': 'Filas Duplicadas Exactas', 'Valor': f'{duplicados_exactos:,}'},
    {'Métrica': 'Uso de Memoria en RAM', 'Valor': f'{memoria_mb:.2f} MB'},
])

print('\n=== RESUMEN GENERAL DEL ARCHIVO ===')
display(resumen_general)

=== METADATOS DEL REPORTE (FILAS PREVIAS A LA CABECERA) ===
  Fila 1: Dataset - Obras Públicas
  Fila 2: Fuente: Sistema de información de obras públicas - INFOBRAS
  Fila 3: Fecha de consulta: 01/10/2025

Cargando datos desde data/raw/cgr/2025/DataSet-Obras-Publicas 01-10-2025.xlsx (Cabecera en fila 4)...

=== RESUMEN GENERAL DEL ARCHIVO ===


,Métrica,Valor
0,Archivo (Ruta Relativa),data/raw/cgr/2025/DataSet-Obras-Publicas 01-10-2025.xlsx
1,Formato / Extensión,.xlsx
2,Fila de Cabecera Utilizada,Fila 4 (skiprows=3)
3,Cantidad Total de Registros (Filas),"184,266"
4,Cantidad Total de Columnas,113
5,Filas Duplicadas Exactas,0
6,Uso de Memoria en RAM,853.51 MB


## 5. Resumen de Tipos de Datos y Completitud

In [22]:
df_profiling = generar_profiling_columnas(df)

print('=== DISTRIBUCIÓN POR TIPO DE DATO PANDAS (dtype) ===')
resumen_dtype = (
    df_profiling.groupby('dtype_pandas', as_index=False)
    .agg(cantidad_columnas=('nro_col', 'count'))
    .sort_values('cantidad_columnas', ascending=False)
)
resumen_dtype['porcentaje (%)'] = (resumen_dtype['cantidad_columnas'] / len(df_profiling) * 100).round(2)
display(resumen_dtype)

print('\n=== DISTRIBUCIÓN POR TIPO DE DATO INFERIDO (CONTENIDO REAL) ===')
resumen_inferido = (
    df_profiling.groupby('tipo_inferido', as_index=False)
    .agg(cantidad_columnas=('nro_col', 'count'))
    .sort_values('cantidad_columnas', ascending=False)
)
resumen_inferido['porcentaje (%)'] = (resumen_inferido['cantidad_columnas'] / len(df_profiling) * 100).round(2)
display(resumen_inferido)

=== DISTRIBUCIÓN POR TIPO DE DATO PANDAS (dtype) ===


,dtype_pandas,cantidad_columnas,porcentaje (%)
2,str,86,76.11
1,int64,16,14.16
0,float64,11,9.73



=== DISTRIBUCIÓN POR TIPO DE DATO INFERIDO (CONTENIDO REAL) ===


,tipo_inferido,cantidad_columnas,porcentaje (%)
7,texto / categórico,52,46.02
6,fecha (almacenada como texto),16,14.16
5,entero (int),16,14.16
4,entero (float por nulos),11,9.73
0,booleano (SI/NO),8,7.08
2,decimal (almacenado como texto),4,3.54
3,entero (almacenado como texto),3,2.65
1,código numérico (texto con ceros izq.),2,1.77
8,vacío (cadenas vacías),1,0.88


## 6. Listado Completo de Columnas y Tipos de Datos

A continuación se listan todas las columnas identificadas en el archivo junto con su posición, nombre original, nombre normalizado, tipo de dato en Pandas, tipo inferido, cantidad de valores no nulos/nulos, cardinalidad y muestra de datos.

In [23]:
df_profiling.nombre_columna

0                                        codigo Entidad
1                                       Entidad Pública
2                                       Código INFOBRAS
3                                        Nombre de obra
4                     Modalidad de ejecución de la obra
5                      ¿Corresponde a un saldo de Obra?
6                       Código INFOBRAS de obra inicial
7                                 Naturaleza de la obra
8                   Tipo de obra - Clasificador Nivel 1
9                   Tipo de obra - Clasificador Nivel 2
10                  Tipo de obra - Clasificador Nivel 3
11                                  Estado de ejecución
12             Marca Reconstrucción con Cambios (Si/No)
13                Marca Reactivación Economicas (Si/No)
14                                           Otra Marca
15                  N° Informes de Monitores Ciudadanos
16                              N° Denuncias vinculadas
17                               N° Informes de 

In [16]:
print(f'Listado de las {len(df_profiling)} columnas del archivo {ARCHIVO_RELATIVO}:')
display(df_profiling)

Listado de las 97 columnas del archivo data/raw/cgr/2026/DataSet-Obras-Publicas 01-10-2026.xlsx:


,nro_col,nombre_columna,columna_normalizada,dtype_pandas,tipo_inferido,total_registros,no_nulos,nulos_o_vacios,pct_nulos (%),valores_unicos,pct_cardinalidad (%),muestra_valores
0,1,codigo Entidad,codigo_entidad,float64,entero (float por nulos),191180,191150,30,0.02,2465,1.29,4338.0 | 608.0 | 1317.0
1,2,Entidad Pública,entidad_publica,str,texto / categórico,191180,191150,30,0.02,2464,1.29,HOSPITAL CHANCAY Y SERVICIOS BÁSICOS DE SALUD | PROYECTO ESPECIAL CHAVIMOCHI...
2,3,Código INFOBRAS,codigo_infobras,int64,entero (int),191180,191180,0,0.00,191180,100.00,4 | 6 | 7
3,4,Nombre de obra,nombre_de_obra,str,texto / categórico,191180,191180,0,0.00,189160,98.94,IMPLEMENTACIÓN DE LA CAPACIDAD RESOLUTIVA DE | CONSTRUCCION DE CANALES INTE...
4,5,Modalidad de ejecución de la obra,modalidad_de_ejecucion_de_la_obra,str,texto / categórico,191180,191180,0,0.00,8,0.00,Administración directa | Contrata | Asociacion Pública Privada APP
5,6,¿Corresponde a un saldo de Obra?,corresponde_a_un_saldo_de_obra,str,booleano (SI/NO),191180,191180,0,0.00,2,0.00,NO | SI
6,7,Código INFOBRAS de obra inicial,codigo_infobras_de_obra_inicial,float64,entero (float por nulos),191180,1283,189897,99.33,1164,90.72,827.0 | 829.0 | 1644.0
7,8,Naturaleza de la obra,naturaleza_de_la_obra,str,texto / categórico,191180,191157,23,0.01,16,0.01,Mejoramiento | Construcción/Creación | Ampliación
8,9,Tipo de obra - Clasificador Nivel 1,tipo_de_obra_clasificador_nivel_1,str,texto / categórico,191180,191180,0,0.00,11,0.01,Salud | Agricultura | Transportes Y Comunicaciones
9,10,Tipo de obra - Clasificador Nivel 2,tipo_de_obra_clasificador_nivel_2,str,texto / categórico,191180,191180,0,0.00,42,0.02,Atención Médica | Riego | Transporte Urbano


## 7. Vista Previa de los Primeros Registros del Dataset

In [7]:
display(df.head(5))

,codigo Entidad,Entidad Pública,Código INFOBRAS,Nombre de obra,Modalidad de ejecución de la obra,¿Corresponde a un saldo de Obra?,Código INFOBRAS de obra inicial,Naturaleza de la obra,Tipo de obra - Clasificador Nivel 1,Tipo de obra - Clasificador Nivel 2,Tipo de obra - Clasificador Nivel 3,Estado de ejecución,Marca Reconstrucción con Cambios (Si/No),Marca Reactivación Economicas (Si/No),Otra Marca,N° Informes de Monitores Ciudadanos,N° Denuncias vinculadas,N° Informes de control,N° Comentarios ciudadanos,¿Es una obra de carácter reservado?,Nivel de gobierno,Sector de la Entidad,Codigo unico de inversión,Código SNIP,Nombre proyecto,...,N° de modificaciones,N° días de modificaciones de plazo,Nuevo Plazo de ejecución (en dias),Fecha finalización reprogramada de obra,N° de controversias,N° de adicionales de obra,Monto de adicionales de obra en soles,N° de adicionales de supervisión,Monto de adicionales de supervisión en soles,N° de deductivos de obra,Monto de deductivos de obra en soles,N° de deductivos de supervisión,Monto de deductivos de supervisión en soles,Fecha de finalización real,Motivo en caso no se llegue al 100%,¿Tiene recepción total?,Fecha de recepción,¿Tiene liquidación de obra?,Fecha de aprobación de liquidación de obra,Costo de la obra en soles,¿Corresponde la transferencia a otra entidad?,Fecha de transferencia,Tipo de entidad que recepciona la obra,Año de primer devengado,Monto Total devengado del proyecto
0,4338,HOSPITAL CHANCAY Y SERVICIOS BÁSICOS DE SALUD UNIDAD EJECUTORA 405,4,IMPLEMENTACIÓN DE LA CAPACIDAD RESOLUTIVA DE LA UNIDAD DE GINECO OBSTETRICIA...,Administración directa,NO,NaN,Mejoramiento,Salud,Atención Médica,Otra Infraestructura,En Ejecución,No,No,NaN,0,NaN,1,0,No,GOBIERNO REGIONAL,SALUD,2111665,110758,IMPLEMENTACION DE LA CAPACIDAD RESOLUTIVA DE LA UNIDAD GINECO OBSTETRICIA 2D...,...,0,0,0,06/10/2011,0,0,0,0,0,0,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,608,PROYECTO ESPECIAL CHAVIMOCHIC,6,CONSTRUCCION DE CANALES INTEGRADORES VALLE VIRU,Contrata,NO,NaN,Construcción/Creación,Agricultura,Riego,Canales,Finalizado,No,No,NaN,0,NaN,0,1,No,GOBIERNO REGIONAL,GOBIERNOS REGIONALES,2106321,109575,CONSTRUCCION DE CANALES INTEGOBIERNO REGIONALADORES VALLE VIRU,...,0,0,0,12/01/2010,0,0,0,0,0,0,0,0,0,10/03/2011,Otros,Si,31/03/2012,Si,29/10/2014,37740948 23,NaN,NaN,NaN,NaN,NaN
2,1317,MUNICIPALIDAD DISTRITAL DE YANAHUARA,7,CULMINACION DE LA PAVIMENTACION DE LA AVENIDA METROPOLITANA CHALLAPAMPA,Contrata,NO,NaN,Construcción/Creación,Transportes Y Comunicaciones,Transporte Urbano,Pista Y Vereda,Finalizado,No,No,NaN,0,NaN,0,0,No,GOBIERNO LOCAL,GOBIERNOS LOCALES,0,0,NaN,...,0,0,15,06/03/2012,0,0,0,0,0,0,0,0,0,05/03/2012,Otros,Si,31/03/2012,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,1317,MUNICIPALIDAD DISTRITAL DE YANAHUARA,8,ALCANTARILLADO PLUVIAL DE LA ZONA TRADICIONAL DE YANAHUARA,Contrata,NO,NaN,Mejoramiento,Vivienda Construcción Y Saneamiento,Saneamiento Urbano Y Rural,Alcantarillado,Finalizado,No,No,NaN,0,NaN,0,0,No,GOBIERNO LOCAL,GOBIERNOS LOCALES,2107907,111215,ALCANTARILLADO PLUVIAL ZONA TRADICIONAL DE YANAHUARA DISTRITO YANAHUARA PROV...,...,0,0,180,24/06/2012,0,0,0,0,0,0,0,0,0,31/08/2012,Otros,Si,31/08/2012,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,608,PROYECTO ESPECIAL CHAVIMOCHIC,9,AMPLIACION DEL SISTEMA DE DISTRIBUCION SECUNDARIA Y MEDIA TENSION DEL CENTRO...,Contrata,NO,NaN,Ampliación,Energía Y Minas,Energía Eléctrica,Sistema De Distribución Eléctrica,Finalizado,No,No,NaN,0,NaN,0,0,No,GOBIERNO REGIONAL,GOBIERNOS REGIONALES,2134526,110028,AMPLIACION DEL SISTEMA DE DISTRIBUCION SECUNDARIA Y MEDIA TENSION DEL CENTRO...,...,0,0,0,06/01/2012,0,0,0,0,0,0,0,0,0,15/03/2012,Otros,Si,31/03/2012,Si,31/05/2012,229430 21,NaN,NaN,NaN,NaN,NaN
